# Multimodal candidate profile visual report

This notebook creates a more presentation-friendly view of your multimodal outputs: candidate profile cards, profile heatmaps, emotion and topic views, correlation graphs, peak-event examples, and interactive explorers.

It expects the files created by your analysis script, usually in `outputs/multimodal_profile_analysis/`:

- `01_multimodal_10s_windows.csv`
- `02_candidate_profiles_global.csv`
- `03_candidate_profiles_by_debate.csv`
- `04_candidate_topic_profiles.csv`
- `05_global_correlations.csv`
- `06_candidate_correlations.csv`
- `07_topic_correlations.csv`
- `08_peak_events.csv`
- `09_analysis_assessments.txt`

All presentation figures are saved to `outputs/presentation_graphs/`.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown, clear_output

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(".").resolve()

candidate_dirs = [
    PROJECT_ROOT / "outputs" / "multimodal_profile_analysis",
    PROJECT_ROOT / "outputs" / "analysis",
    PROJECT_ROOT / "analysis_outputs",
    PROJECT_ROOT / "outputs" / "analysis_outputs",
]

ANALYSIS_DIR = None
for p in candidate_dirs:
    if (p / "02_candidate_profiles_global.csv").exists():
        ANALYSIS_DIR = p
        break

if ANALYSIS_DIR is None:
    print("Could not find analysis outputs automatically. Checked:")
    for p in candidate_dirs:
        print(" -", p)
    print("\nEdit ANALYSIS_DIR manually, for example:")
    print("ANALYSIS_DIR = Path('outputs/multimodal_profile_analysis')")
    raise FileNotFoundError("Analysis outputs not found")

PLOT_DIR = PROJECT_ROOT / "outputs" / "presentation_graphs"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Analysis directory:", ANALYSIS_DIR)
print("Plot folder:", PLOT_DIR)

In [ ]:
def read_csv(name):
    path = ANALYSIS_DIR / name
    if not path.exists():
        print("Missing:", path)
        return pd.DataFrame()
    return pd.read_csv(path)

multimodal = read_csv("01_multimodal_10s_windows.csv")
profiles = read_csv("02_candidate_profiles_global.csv")
profiles_by_debate = read_csv("03_candidate_profiles_by_debate.csv")
topic_profiles = read_csv("04_candidate_topic_profiles.csv")
global_corr = read_csv("05_global_correlations.csv")
candidate_corr = read_csv("06_candidate_correlations.csv")
topic_corr = read_csv("07_topic_correlations.csv")
peak_events = read_csv("08_peak_events.csv")

assessment_path = ANALYSIS_DIR / "09_analysis_assessments.txt"
assessment_text = assessment_path.read_text(encoding="utf-8") if assessment_path.exists() else ""

MODERATOR_LABELS = {"Moderador/Other", "Moderator/Other", "No speech", "nan", "None"}

def is_real_candidate(series):
    return ~series.astype(str).isin(MODERATOR_LABELS)

profiles_c = profiles[is_real_candidate(profiles["candidate"])].copy()
profiles_by_debate_c = profiles_by_debate[is_real_candidate(profiles_by_debate["candidate"])].copy()
topic_profiles_c = topic_profiles[is_real_candidate(topic_profiles["candidate"])].copy()
multimodal_c = multimodal[is_real_candidate(multimodal["candidate"])].copy() if "candidate" in multimodal.columns else multimodal.copy()
peak_events_c = peak_events[is_real_candidate(peak_events["candidate"])].copy() if "candidate" in peak_events.columns else peak_events.copy()

print("Loaded files:")
for name, df in {
    "multimodal": multimodal,
    "profiles": profiles,
    "profiles_by_debate": profiles_by_debate,
    "topic_profiles": topic_profiles,
    "global_corr": global_corr,
    "candidate_corr": candidate_corr,
    "topic_corr": topic_corr,
    "peak_events": peak_events,
}.items():
    print(f"{name:22s}", df.shape)

display(profiles_c.head())

## Helper functions

In [ ]:
def save_plot(filename):
    path = PLOT_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight")
    print("Saved:", path)


def norm01(s):
    s = pd.to_numeric(s, errors="coerce")
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn)


def pretty(name):
    return (
        str(name)
        .replace("avg_", "")
        .replace("mean_", "")
        .replace("_", " ")
        .replace("speechrate", "speech rate")
        .replace("pitchvar", "pitch variation")
        .replace("non neutral", "non-neutral")
        .title()
    )


def top_items_from_string(s, n=4):
    if pd.isna(s):
        return []
    parts = [p.strip() for p in str(s).split(";")]
    return [p for p in parts if p][:n]


def heatmap(pivot, title, cbar_label, filename, figsize=(12,7), annotate=False, fmt=".2f"):
    data = pivot.copy()
    values = data.fillna(0).to_numpy(dtype=float)
    plt.figure(figsize=figsize)
    plt.imshow(values, aspect="auto")
    plt.title(title)
    plt.xticks(range(len(data.columns)), data.columns, rotation=45, ha="right")
    plt.yticks(range(len(data.index)), data.index)
    plt.xlabel(data.columns.name or "")
    plt.ylabel(data.index.name or "")
    plt.colorbar(label=cbar_label)
    if annotate and values.shape[0] * values.shape[1] <= 120:
        for i in range(values.shape[0]):
            for j in range(values.shape[1]):
                plt.text(j, i, format(values[i, j], fmt), ha="center", va="center", fontsize=7)
    save_plot(filename)
    plt.show()


def metric(row, col, digits=3):
    if col not in row.index or pd.isna(row[col]):
        return "n/a"
    return f"{row[col]:.{digits}f}"

# 1. Candidate profile cards

These cards combine the main measures into a readable profile: movement, hand gestures, non-neutral emotion, speech rate, pitch variation, interruptions, top emotions, and top topics.

In [ ]:
STYLE_FEATURES = {
    "Movement": "avg_movement",
    "Hand gestures": "avg_hand_movement",
    "Emotion intensity": "avg_non_neutral_emotion_score",
    "Speech rate": "avg_speechrate_z",
    "Pitch variation": "avg_pitchvar_z",
    "Interruptions": "interruptions_per_100_speaking_sec",
}
STYLE_FEATURES = {label: col for label, col in STYLE_FEATURES.items() if col in profiles_c.columns}

style_df = profiles_c[["candidate"] + list(STYLE_FEATURES.values())].copy()
for label, col in STYLE_FEATURES.items():
    style_df[label] = norm01(style_df[col])
style_norm = style_df[["candidate"] + list(STYLE_FEATURES.keys())].set_index("candidate")

def badge(text):
    return f"<span class='badge'>{text}</span>"


def candidate_badges(style_row):
    out = []
    if style_row.get("Movement", 0) >= 0.70:
        out.append("high movement")
    if style_row.get("Hand gestures", 0) >= 0.70:
        out.append("gestural")
    if style_row.get("Emotion intensity", 0) >= 0.70:
        out.append("visually expressive")
    if style_row.get("Speech rate", 0) >= 0.70:
        out.append("fast speech")
    if style_row.get("Pitch variation", 0) >= 0.70:
        out.append("variable pitch")
    if style_row.get("Interruptions", 0) >= 0.70:
        out.append("interrupts often")
    if not out:
        out.append("more composed / balanced")
    return out


def mini_bar(label, value):
    value = 0 if pd.isna(value) else float(value)
    pct = max(0, min(100, value * 100))
    return f"""
    <div class='bar-row'>
      <div class='bar-label'>{label}</div>
      <div class='bar-track'><div class='bar-fill' style='width:{pct:.0f}%'></div></div>
      <div class='bar-num'>{pct:.0f}</div>
    </div>
    """

cards = []
for _, row in profiles_c.iterrows():
    cand = row["candidate"]
    if cand not in style_norm.index:
        continue
    sr = style_norm.loc[cand]
    badges = " ".join(badge(x) for x in candidate_badges(sr))
    bars = "".join(mini_bar(label, sr[label]) for label in style_norm.columns)
    top_emotions = "<br>".join(top_items_from_string(row.get("top_emotions", ""), 4)) or "n/a"
    top_topics = "<br>".join(top_items_from_string(row.get("top_topics", ""), 4)) or "n/a"
    n_debates = int(row.get("n_debates", 0)) if pd.notna(row.get("n_debates", np.nan)) else "n/a"
    cards.append(f"""
    <div class='profile-card'>
      <h3>{cand}</h3>
      <div>{badges}</div>
      <div class='quick-stats'>
        <div><b>Debates</b><br>{n_debates}</div>
        <div><b>Speaking sec.</b><br>{metric(row, 'total_speaking_seconds', 0)}</div>
        <div><b>Visible sec.</b><br>{metric(row, 'total_visible_seconds', 0)}</div>
      </div>
      {bars}
      <div class='two-col'>
        <div><b>Top emotions</b><br>{top_emotions}</div>
        <div><b>Top topics</b><br>{top_topics}</div>
      </div>
    </div>
    """)

html = f"""
<style>
.profile-grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(320px,1fr)); gap:16px; }}
.profile-card {{ border:1px solid #ddd; border-radius:14px; padding:16px; box-shadow:0 2px 8px rgba(0,0,0,.08); background:white; }}
.profile-card h3 {{ margin:0 0 8px 0; }}
.badge {{ display:inline-block; border:1px solid #bbb; border-radius:999px; padding:3px 8px; margin:2px; font-size:12px; background:#f7f7f7; }}
.quick-stats {{ display:grid; grid-template-columns:repeat(3,1fr); gap:8px; margin:12px 0; font-size:13px; }}
.quick-stats div {{ border-radius:10px; background:#f6f6f6; padding:8px; }}
.bar-row {{ display:grid; grid-template-columns:110px 1fr 34px; align-items:center; gap:8px; margin:5px 0; font-size:12px; }}
.bar-track {{ background:#e9e9e9; border-radius:999px; height:10px; overflow:hidden; }}
.bar-fill {{ background:#777; height:100%; border-radius:999px; }}
.bar-num {{ text-align:right; color:#555; }}
.two-col {{ display:grid; grid-template-columns:1fr 1fr; gap:12px; margin-top:12px; font-size:13px; line-height:1.35; }}
</style>
<div class='profile-grid'>{''.join(cards)}</div>
"""

display(HTML(html))

# Save a standalone HTML version for easy screenshots.
html_path = PLOT_DIR / "candidate_profile_cards.html"
html_path.write_text(f"<html><body><h1>Candidate debating style profiles</h1>{html}</body></html>", encoding="utf-8")
print("Saved profile-card HTML:", html_path)

# 2. Candidate profile graphs

In [ ]:
# Style heatmap
heatmap(
    style_norm.round(3),
    title="Candidate debating style profile",
    cbar_label="Normalized score",
    filename="01_candidate_style_profile_heatmap.png",
    figsize=(12, max(5, 0.55 * len(style_norm))),
    annotate=True,
)

In [ ]:
# Movement profile
movement_cols = [c for c in ["avg_movement", "avg_hand_movement", "avg_pose_movement"] if c in profiles_c.columns]
move_df = profiles_c[["candidate"] + movement_cols].copy().sort_values("avg_movement", ascending=False)

plt.figure(figsize=(12, 6))
x = np.arange(len(move_df))
bar_w = 0.8 / max(1, len(movement_cols))
for i, col in enumerate(movement_cols):
    plt.bar(x + i * bar_w, move_df[col], width=bar_w, label=pretty(col))
plt.xticks(x + bar_w * (len(movement_cols)-1) / 2, move_df["candidate"], rotation=45, ha="right")
plt.title("Average visual movement by candidate")
plt.ylabel("Average movement score")
plt.xlabel("Candidate")
plt.legend()
save_plot("02_average_movement_by_candidate.png")
plt.show()

In [ ]:
# Emotion accumulation: heatmap + stacked bars
emotion_cols = [c for c in profiles_c.columns if c.startswith("emotion_avg_share_")]
if emotion_cols:
    emotion_profile = profiles_c[["candidate"] + emotion_cols].set_index("candidate")
    emotion_profile.columns = [c.replace("emotion_avg_share_", "") for c in emotion_profile.columns]

    heatmap(
        emotion_profile.round(3),
        title="Accumulated emotion profile by candidate",
        cbar_label="Average emotion share",
        filename="03_emotion_profile_by_candidate_heatmap.png",
        figsize=(12, max(5, 0.55 * len(emotion_profile))),
        annotate=True,
    )

    emotion_profile.plot(kind="bar", stacked=True, figsize=(14, 6))
    plt.title("Emotion composition by candidate")
    plt.ylabel("Average emotion share")
    plt.xlabel("Candidate")
    plt.xticks(rotation=45, ha="right")
    plt.legend(title="Emotion", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_plot("04_emotion_composition_by_candidate.png")
    plt.show()
else:
    print("No emotion_avg_share_ columns found.")

In [ ]:
# Audio behavior profile
cols = [c for c in ["avg_speechrate_z", "avg_pitchvar_z", "interruptions_per_100_speaking_sec"] if c in profiles_c.columns]
if cols:
    audio_profile = profiles_c[["candidate"] + cols].copy()
    for c in cols:
        audio_profile[pretty(c)] = norm01(audio_profile[c])
    plot_cols = [pretty(c) for c in cols]
    audio_profile = audio_profile.set_index("candidate")[plot_cols]
    audio_profile.plot(kind="bar", figsize=(12, 5))
    plt.title("Audio behavior profile by candidate")
    plt.ylabel("Normalized score")
    plt.xlabel("Candidate")
    plt.xticks(rotation=45, ha="right")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    save_plot("05_audio_behavior_profile_by_candidate.png")
    plt.show()
else:
    print("Audio profile columns not found.")

In [ ]:
# Expressiveness map: movement vs non-neutral emotion, size = interruptions
x_col = "avg_movement"
y_col = "avg_non_neutral_emotion_score"
size_col = "interruptions_per_100_speaking_sec"

if {x_col, y_col}.issubset(profiles_c.columns):
    plot_df = profiles_c.copy()
    sizes = 180
    if size_col in plot_df.columns:
        sizes = 80 + 500 * norm01(plot_df[size_col])
    plt.figure(figsize=(10, 7))
    plt.scatter(plot_df[x_col], plot_df[y_col], s=sizes, alpha=0.75)
    for _, r in plot_df.iterrows():
        plt.text(r[x_col], r[y_col], str(r["candidate"]), fontsize=9, ha="left", va="bottom")
    plt.title("Candidate expressiveness map")
    plt.xlabel("Average movement")
    plt.ylabel("Average non-neutral emotion score")
    plt.grid(True, alpha=0.3)
    save_plot("06_candidate_expressiveness_map.png")
    plt.show()
else:
    print("Required columns missing for expressiveness map.")

# 3. Topic profiles

These heatmaps show which topics are associated with more visual emotion and movement for each candidate.

In [ ]:
if not topic_profiles_c.empty:
    top_topics = (
        topic_profiles_c.groupby("topic")["topic_overlap_seconds"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .index
    )
    topic_plot = topic_profiles_c[topic_profiles_c["topic"].isin(top_topics)].copy()

    if "avg_non_neutral_emotion_score" in topic_plot.columns:
        topic_emotion = topic_plot.pivot_table(
            index="topic", columns="candidate", values="avg_non_neutral_emotion_score", aggfunc="mean"
        )
        heatmap(topic_emotion, "Average non-neutral emotion by topic and candidate", "Avg non-neutral emotion", "07_topic_candidate_emotion_heatmap.png", figsize=(14, 8), annotate=True)

    if "avg_movement" in topic_plot.columns:
        topic_movement = topic_plot.pivot_table(
            index="topic", columns="candidate", values="avg_movement", aggfunc="mean"
        )
        heatmap(topic_movement, "Average movement by topic and candidate", "Avg movement", "08_topic_candidate_movement_heatmap.png", figsize=(14, 8), annotate=True, fmt=".3f")

    interesting_cols = [
        "candidate", "topic", "topic_overlap_seconds", "avg_movement", "avg_hand_movement",
        "avg_non_neutral_emotion_score", "avg_speechrate_z", "avg_pitchvar_z",
        "interruption_proxy_seconds", "dominant_emotion_mode",
    ]
    interesting_cols = [c for c in interesting_cols if c in topic_profiles_c.columns]
    display(HTML("<h3>Most visually expressive candidate-topic combinations</h3>"))
    display(topic_profiles_c.sort_values(["avg_non_neutral_emotion_score", "avg_movement"], ascending=False)[interesting_cols].head(20))
else:
    print("No topic profile data available.")

# 4. Correlation examples

In [ ]:
# Strongest global correlations
if not global_corr.empty:
    corr = global_corr.copy()
    corr["pair"] = corr["x"].map(pretty) + " vs " + corr["y"].map(pretty)
    corr = corr.dropna(subset=["spearman"]).sort_values("spearman", key=lambda s: s.abs(), ascending=False).head(15)

    plt.figure(figsize=(11, 7))
    plt.barh(corr["pair"][::-1], corr["spearman"][::-1])
    plt.title("Strongest multimodal correlations")
    plt.xlabel("Spearman correlation")
    plt.axvline(0, linewidth=1)
    save_plot("09_strongest_global_correlations.png")
    plt.show()

    display(corr[["x", "y", "n", "pearson", "spearman"]])
else:
    print("No global correlations available.")

In [ ]:
# Scatter: movement vs speech rate
if {"mean_movement", "mean_speechrate_z", "candidate"}.issubset(multimodal_c.columns):
    scatter_df = multimodal_c[multimodal_c["mean_movement"].notna() & multimodal_c["mean_speechrate_z"].notna()].copy()
    if len(scatter_df) > 3000:
        scatter_df = scatter_df.sample(3000, random_state=42)
    plt.figure(figsize=(10, 7))
    for candidate, g in scatter_df.groupby("candidate"):
        plt.scatter(g["mean_movement"], g["mean_speechrate_z"], alpha=0.35, label=candidate, s=20)
    plt.title("Movement vs speech rate across 10-second windows")
    plt.xlabel("Mean visual movement")
    plt.ylabel("Speech rate z-score")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    save_plot("10_movement_vs_speech_rate_scatter.png")
    plt.show()
else:
    print("Required columns not found for movement vs speech rate scatter.")

In [ ]:
# Scatter: emotion intensity vs pitch variation
if {"mean_non_neutral_emotion_score", "mean_pitchvar_z", "candidate"}.issubset(multimodal_c.columns):
    scatter_df = multimodal_c[multimodal_c["mean_non_neutral_emotion_score"].notna() & multimodal_c["mean_pitchvar_z"].notna()].copy()
    if len(scatter_df) > 3000:
        scatter_df = scatter_df.sample(3000, random_state=42)
    plt.figure(figsize=(10, 7))
    for candidate, g in scatter_df.groupby("candidate"):
        plt.scatter(g["mean_non_neutral_emotion_score"], g["mean_pitchvar_z"], alpha=0.35, label=candidate, s=20)
    plt.title("Emotion intensity vs pitch variation")
    plt.xlabel("Mean non-neutral emotion score")
    plt.ylabel("Pitch variation z-score")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    save_plot("11_emotion_intensity_vs_pitch_variation.png")
    plt.show()
else:
    print("Required columns not found for emotion vs pitch scatter.")

In [ ]:
# Candidate-specific correlations: top 3 per candidate
if not candidate_corr.empty:
    cc = candidate_corr.dropna(subset=["spearman"]).copy()
    cc["pair"] = cc["x"].map(pretty) + " vs " + cc["y"].map(pretty)
    strongest = (
        cc.assign(abs_spearman=cc["spearman"].abs())
        .sort_values(["group", "abs_spearman"], ascending=[True, False])
        .groupby("group")
        .head(3)
        [["group", "pair", "n", "spearman"]]
    )
    display(HTML("<h3>Top candidate-specific correlations</h3>"))
    display(strongest)
else:
    print("No candidate correlation data available.")

# 5. Peak event examples

These moments are useful for slides because they show windows where several modalities peaked at the same time.

In [ ]:
if not peak_events_c.empty:
    counts = peak_events_c.groupby(["multimodal_event_type", "candidate"]).size().reset_index(name="count")
    top_types = peak_events_c["multimodal_event_type"].value_counts().head(8).index
    pivot = counts[counts["multimodal_event_type"].isin(top_types)].pivot_table(
        index="multimodal_event_type", columns="candidate", values="count", fill_value=0, aggfunc="sum"
    )
    heatmap(pivot, "Peak multimodal event counts by candidate", "Event count", "12_peak_event_counts_by_candidate.png", figsize=(14, 6), annotate=True, fmt=".0f")

    display_cols = [
        "multimodal_event_type", "debate_name", "candidate", "window_start", "window_end",
        "dominant_topic", "dominant_emotion", "mean_movement", "mean_hand_movement",
        "mean_non_neutral_emotion_score", "mean_speechrate_z", "mean_pitchvar_z",
        "speaking_seconds", "visible_seconds", "interruption_proxy_seconds",
    ]
    display_cols = [c for c in display_cols if c in peak_events_c.columns]
    display(HTML("<h3>Top peak event examples to inspect manually</h3>"))
    display(peak_events_c[display_cols].head(25))
else:
    print("No peak events available.")

# 6. Interactive candidate explorer

Pick a candidate to see their profile, emotions, most discussed topics, and debate-by-debate variation.

In [ ]:
try:
    import ipywidgets as widgets

    candidate_options = sorted(profiles_c["candidate"].dropna().astype(str).unique())
    candidate_dropdown = widgets.Dropdown(options=candidate_options, description="Candidate:", layout=widgets.Layout(width="60%"))
    candidate_output = widgets.Output()

    def show_candidate(change=None):
        cand = candidate_dropdown.value
        with candidate_output:
            clear_output(wait=True)
            row_df = profiles_c[profiles_c["candidate"] == cand]
            if row_df.empty:
                print("Candidate not found")
                return
            row = row_df.iloc[0]
            display(HTML(f"<h2>{cand}</h2>"))
            display(Markdown(f"""
**Debates:** {row.get('n_debates', 'n/a')}  
**Total speaking seconds:** {row.get('total_speaking_seconds', np.nan):.0f}  
**Total visible seconds:** {row.get('total_visible_seconds', np.nan):.0f}  
**Top emotions:** {row.get('top_emotions', 'n/a')}  
**Top topics:** {row.get('top_topics', 'n/a')}
"""))

            if cand in style_norm.index:
                sr = style_norm.loc[cand].sort_values()
                plt.figure(figsize=(8, 4))
                plt.barh(sr.index, sr.values)
                plt.title(f"Style dimensions — {cand}")
                plt.xlabel("Normalized score")
                plt.xlim(0, 1)
                plt.tight_layout()
                plt.show()

            if emotion_cols:
                emo = profiles_c[profiles_c["candidate"] == cand][emotion_cols].iloc[0]
                emo.index = [c.replace("emotion_avg_share_", "") for c in emo.index]
                emo = emo.sort_values()
                plt.figure(figsize=(8, 4))
                plt.barh(emo.index, emo.values)
                plt.title(f"Emotion accumulation — {cand}")
                plt.xlabel("Average emotion share")
                plt.tight_layout()
                plt.show()

            tdf = topic_profiles_c[topic_profiles_c["candidate"] == cand].copy()
            if not tdf.empty and "topic_overlap_seconds" in tdf.columns:
                tdf = tdf.sort_values("topic_overlap_seconds", ascending=False).head(8)
                plt.figure(figsize=(9, 4))
                plt.barh(tdf["topic"][::-1], tdf["topic_overlap_seconds"][::-1])
                plt.title(f"Most discussed topics — {cand}")
                plt.xlabel("Topic overlap seconds")
                plt.tight_layout()
                plt.show()

            bdf = profiles_by_debate_c[profiles_by_debate_c["candidate"] == cand].copy()
            if not bdf.empty:
                cols = [c for c in ["avg_movement", "avg_non_neutral_emotion_score", "avg_speechrate_z"] if c in bdf.columns]
                if cols:
                    bdf = bdf.sort_values("debate_name")
                    plot_bdf = bdf[["debate_name"] + cols].copy()
                    for c in cols:
                        plot_bdf[pretty(c)] = norm01(plot_bdf[c])
                    plot_cols = [pretty(c) for c in cols]
                    plot_bdf = plot_bdf.set_index("debate_name")[plot_cols]
                    plot_bdf.plot(kind="bar", figsize=(12, 4))
                    plt.title(f"Behavior by debate — {cand}")
                    plt.ylabel("Normalized within candidate")
                    plt.xticks(rotation=45, ha="right")
                    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
                    plt.tight_layout()
                    plt.show()

    candidate_dropdown.observe(show_candidate, names="value")
    display(widgets.VBox([candidate_dropdown, candidate_output]))
    show_candidate()
except Exception as e:
    print("Interactive candidate explorer failed:", repr(e))

# 7. Interactive timeline explorer

Pick a debate and candidate to inspect how movement, emotion, speech rate, and pitch variation evolve over time.

In [ ]:
try:
    import ipywidgets as widgets

    if not multimodal_c.empty:
        debate_options = sorted(multimodal_c["debate_name"].dropna().astype(str).unique())
        debate_dropdown = widgets.Dropdown(options=debate_options, description="Debate:", layout=widgets.Layout(width="80%"))
        cand_dropdown = widgets.Dropdown(description="Candidate:", layout=widgets.Layout(width="60%"))
        timeline_output = widgets.Output()

        def update_candidates(change=None):
            debate = debate_dropdown.value
            opts = sorted(multimodal_c[multimodal_c["debate_name"] == debate]["candidate"].dropna().astype(str).unique())
            cand_dropdown.options = opts
            if opts:
                cand_dropdown.value = opts[0]

        def show_timeline(change=None):
            debate = debate_dropdown.value
            cand = cand_dropdown.value
            if cand is None:
                return
            with timeline_output:
                clear_output(wait=True)
                df = multimodal_c[(multimodal_c["debate_name"] == debate) & (multimodal_c["candidate"] == cand)].copy().sort_values("window_start")
                display(HTML(f"<h3>{cand} — {debate}</h3>"))
                if df.empty:
                    print("No data for this selection")
                    return

                plot_cols = [c for c in ["mean_movement", "mean_non_neutral_emotion_score", "mean_speechrate_z", "mean_pitchvar_z"] if c in df.columns]
                plt.figure(figsize=(14, 5))
                for c in plot_cols:
                    plt.plot(df["window_start"], norm01(df[c]), marker="o", markersize=2, linewidth=1.5, label=pretty(c))
                plt.title("Timeline of normalized multimodal signals")
                plt.xlabel("Window start second")
                plt.ylabel("Normalized inside selected candidate/debate")
                plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()

                if "dominant_topic" in df.columns:
                    topic_counts = df.dropna(subset=["dominant_topic"]).groupby("dominant_topic").size().sort_values(ascending=False).head(8)
                    if len(topic_counts):
                        plt.figure(figsize=(10, 4))
                        plt.barh(topic_counts.index[::-1], topic_counts.values[::-1])
                        plt.title("Most frequent dominant topics in these windows")
                        plt.xlabel("Window count")
                        plt.tight_layout()
                        plt.show()

                rank_score = pd.Series(0.0, index=df.index)
                for c in plot_cols:
                    rank_score += norm01(df[c]).fillna(0)
                df["local_peak_score"] = rank_score
                peak_cols = ["window_start", "window_end", "dominant_topic", "dominant_emotion", "mean_movement", "mean_non_neutral_emotion_score", "mean_speechrate_z", "mean_pitchvar_z", "interruption_proxy_seconds"]
                peak_cols = [c for c in peak_cols if c in df.columns]
                display(HTML("<h4>Most intense local windows</h4>"))
                display(df.sort_values("local_peak_score", ascending=False)[peak_cols + ["local_peak_score"]].head(10))

        debate_dropdown.observe(update_candidates, names="value")
        debate_dropdown.observe(show_timeline, names="value")
        cand_dropdown.observe(show_timeline, names="value")
        update_candidates()
        display(widgets.VBox([debate_dropdown, cand_dropdown, timeline_output]))
        show_timeline()
    else:
        print("No multimodal rows available.")
except Exception as e:
    print("Interactive timeline explorer failed:", repr(e))

# 8. Quick written highlights

Use these as a starting point. Verify the claims with the plots before putting them in the report.

In [ ]:
def top_candidate_by(col, label):
    if col not in profiles_c.columns:
        return None
    df = profiles_c[["candidate", col]].dropna().sort_values(col, ascending=False)
    if df.empty:
        return None
    r = df.iloc[0]
    return f"- **{label}:** {r['candidate']} ({r[col]:.3f})"

summary_lines = []
for col, label in [
    ("avg_movement", "Highest average movement"),
    ("avg_hand_movement", "Highest average hand/arm movement"),
    ("avg_non_neutral_emotion_score", "Highest non-neutral emotion intensity"),
    ("avg_speechrate_z", "Fastest speech relative to average"),
    ("avg_pitchvar_z", "Highest pitch variability"),
    ("interruptions_per_100_speaking_sec", "Most interruptions per 100 speaking seconds"),
]:
    line = top_candidate_by(col, label)
    if line:
        summary_lines.append(line)

if summary_lines:
    display(Markdown("## Quick profile highlights\n" + "\n".join(summary_lines)))

if assessment_text:
    display(Markdown("## Automatic assessment text\n" + assessment_text))
else:
    print("No automatic assessment text found.")

print("All generated plots were saved to:", PLOT_DIR)

In [ ]:
# ============================================================
# CLEAN TOPIC NAMES + REMOVE "NO TOPIC IDENTIFIED"
# Add this at the bottom of the notebook
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap
from IPython.display import display, HTML

OUT_DIR = Path("outputs/multimodal_profile_analysis")
CLEAN_PLOT_DIR = Path("outputs/presentation_graphs_cleaned")
CLEAN_PLOT_DIR.mkdir(parents=True, exist_ok=True)

profiles = pd.read_csv(OUT_DIR / "02_candidate_profiles_global.csv")
topic_profiles = pd.read_csv(OUT_DIR / "04_candidate_topic_profiles.csv")
merged = pd.read_csv(OUT_DIR / "01_multimodal_10s_windows.csv")

print("Loaded:")
print("profiles:", profiles.shape)
print("topic_profiles:", topic_profiles.shape)
print("merged:", merged.shape)


def clean_topic_name(topic):
    """
    Make topic names presentation-friendly.
    Returns None for non-informative/no-topic labels.
    """
    if pd.isna(topic):
        return None

    t = str(topic).strip()

    if t == "":
        return None

    # Normalize ugly separators
    t_clean = (
        t.replace("_", " ")
         .replace("  ", " ")
         .strip()
    )

    # Lowercase version for filtering
    low = (
        t_clean.lower()
        .replace("ã", "a")
        .replace("á", "a")
        .replace("à", "a")
        .replace("â", "a")
        .replace("é", "e")
        .replace("ê", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("õ", "o")
        .replace("ô", "o")
        .replace("ú", "u")
        .replace("ç", "c")
    )

    no_topic_patterns = [
        "nenhum",
        "nao identificado",
        "não identificado",
        "sem topico",
        "sem tema",
        "no topic",
        "unknown",
        "none",
    ]

    if any(p in low for p in no_topic_patterns):
        return None

    # Optional pretty-name replacements
    replacements = {
        "Economia Proteção Social": "Economia / Proteção Social",
        "Economia/Proteção Social": "Economia / Proteção Social",
        "Governo Partidos": "Governo / Partidos",
        "Governo/Partidos": "Governo / Partidos",
        "Eleições Campanha": "Eleições / Campanha",
        "Eleições/Campanha": "Eleições / Campanha",
        "Justiça Segurança": "Justiça / Segurança",
        "Justiça/Segurança": "Justiça / Segurança",
    }

    return replacements.get(t_clean, t_clean)


# Detect topic column in topic_profiles
if "topic" in topic_profiles.columns:
    TOPIC_COL = "topic"
elif "dominant_topic" in topic_profiles.columns:
    TOPIC_COL = "dominant_topic"
else:
    possible_topic_cols = [c for c in topic_profiles.columns if "topic" in c.lower() or "tema" in c.lower()]
    print("Possible topic columns:", possible_topic_cols)
    raise ValueError("Could not find topic column in topic_profiles.")

topic_profiles_clean = topic_profiles.copy()
topic_profiles_clean["topic_clean"] = topic_profiles_clean[TOPIC_COL].apply(clean_topic_name)
topic_profiles_clean = topic_profiles_clean[topic_profiles_clean["topic_clean"].notna()].copy()

# Also clean topic names in merged table
merged_clean = merged.copy()

if "dominant_topic" in merged_clean.columns:
    merged_clean["dominant_topic_clean"] = merged_clean["dominant_topic"].apply(clean_topic_name)
    merged_clean = merged_clean[merged_clean["dominant_topic_clean"].notna()].copy()

print("Original topic rows:", len(topic_profiles))
print("Cleaned topic rows:", len(topic_profiles_clean))

print("\nTopics kept:")
display(
    pd.DataFrame(
        sorted(topic_profiles_clean["topic_clean"].dropna().unique()),
        columns=["topic_clean"]
    )
)

# Save cleaned versions
topic_profiles_clean.to_csv(CLEAN_PLOT_DIR / "candidate_topic_profiles_cleaned.csv", index=False)
merged_clean.to_csv(CLEAN_PLOT_DIR / "multimodal_10s_windows_topics_cleaned.csv", index=False)

In [ ]:
# ============================================================
# CLEANED CANDIDATE PROFILE CARDS
# Better topic names + removes no-topic labels
# ============================================================

def fmt_num(x, digits=3):
    if pd.isna(x):
        return "N/A"
    return f"{x:.{digits}f}"


def get_col(df, possible_names):
    for c in possible_names:
        if c in df.columns:
            return c
    return None


movement_col = get_col(profiles, ["avg_movement", "mean_movement"])
hand_col = get_col(profiles, ["avg_hand_movement", "mean_hand_movement"])
emotion_col = get_col(profiles, ["avg_non_neutral_emotion_score", "mean_non_neutral_emotion_score"])
speech_col = get_col(profiles, ["avg_speechrate_z", "mean_speechrate_z", "avg_speech_rate_z"])
pitch_col = get_col(profiles, ["avg_pitchvar_z", "mean_pitchvar_z", "avg_pitch_z"])
interrupt_col = get_col(profiles, ["interruptions_per_100_speaking_sec", "interruption_rate"])


def top_topics_for_candidate(candidate, n=3):
    sub = topic_profiles_clean[topic_profiles_clean["candidate"] == candidate].copy()

    if len(sub) == 0:
        return []

    # Pick best available weight column
    if "total_topic_overlap_seconds" in sub.columns:
        weight_col = "total_topic_overlap_seconds"
    elif "n_windows" in sub.columns:
        weight_col = "n_windows"
    elif "visible_seconds" in sub.columns:
        weight_col = "visible_seconds"
    else:
        weight_col = None

    if weight_col is not None:
        ranked = (
            sub.groupby("topic_clean")[weight_col]
            .sum()
            .sort_values(ascending=False)
            .head(n)
        )
    else:
        ranked = (
            sub["topic_clean"]
            .value_counts()
            .head(n)
        )

    return ranked.index.tolist()


def top_emotions_for_candidate(row, n=3):
    emotion_cols = [c for c in profiles.columns if c.startswith("emotion_avg_share_")]

    if len(emotion_cols) == 0:
        return []

    values = []

    for c in emotion_cols:
        emotion_name = c.replace("emotion_avg_share_", "")
        val = row.get(c, np.nan)

        if pd.notna(val):
            values.append((emotion_name, val))

    values = sorted(values, key=lambda x: x[1], reverse=True)
    return values[:n]


def candidate_badges(row):
    badges = []

    if movement_col and pd.notna(row.get(movement_col)):
        if row[movement_col] >= profiles[movement_col].quantile(0.75):
            badges.append("high movement")
        elif row[movement_col] <= profiles[movement_col].quantile(0.25):
            badges.append("low movement")

    if hand_col and pd.notna(row.get(hand_col)):
        if row[hand_col] >= profiles[hand_col].quantile(0.75):
            badges.append("gestural")

    if emotion_col and pd.notna(row.get(emotion_col)):
        if row[emotion_col] >= profiles[emotion_col].quantile(0.75):
            badges.append("visually expressive")
        elif row[emotion_col] <= profiles[emotion_col].quantile(0.25):
            badges.append("more composed")

    if speech_col and pd.notna(row.get(speech_col)):
        if row[speech_col] >= profiles[speech_col].quantile(0.75):
            badges.append("fast speaker")
        elif row[speech_col] <= profiles[speech_col].quantile(0.25):
            badges.append("slower speaker")

    if interrupt_col and pd.notna(row.get(interrupt_col)):
        if row[interrupt_col] >= profiles[interrupt_col].quantile(0.75):
            badges.append("frequent interruptions")

    return badges


cards_html = """
<style>
.profile-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(310px, 1fr));
    gap: 16px;
    margin-top: 10px;
}
.profile-card {
    border: 1px solid #ddd;
    border-radius: 14px;
    padding: 16px;
    background: #fafafa;
    box-shadow: 0 1px 4px rgba(0,0,0,0.08);
}
.profile-card h3 {
    margin-top: 0;
    margin-bottom: 8px;
}
.badge {
    display: inline-block;
    padding: 4px 8px;
    margin: 2px;
    border-radius: 999px;
    background: #e8eefc;
    font-size: 12px;
}
.metric {
    margin: 4px 0;
    font-size: 13px;
}
.section-title {
    font-weight: bold;
    margin-top: 10px;
    margin-bottom: 4px;
}
.small {
    font-size: 12px;
    color: #555;
}
</style>

<h2>Candidate profiles with cleaned themes</h2>
<p class="small">Themes labelled as no topic / not identified were removed from the top-topic summaries.</p>
<div class="profile-grid">
"""

for _, row in profiles.sort_values("candidate").iterrows():
    candidate = row["candidate"]

    if candidate in ["Moderador/Other", "No speech"]:
        continue

    badges = candidate_badges(row)
    top_topics = top_topics_for_candidate(candidate, n=3)
    top_emotions = top_emotions_for_candidate(row, n=3)

    badge_html = " ".join([f"<span class='badge'>{b}</span>" for b in badges]) or "<span class='badge'>balanced profile</span>"

    topics_html = "<br>".join([f"• {t}" for t in top_topics]) if top_topics else "No clear topic data"

    emotions_html = "<br>".join([
        f"• {emo}: {val:.1%}"
        for emo, val in top_emotions
    ]) if top_emotions else "No emotion-share data"

    cards_html += f"""
    <div class="profile-card">
        <h3>{candidate}</h3>
        <div>{badge_html}</div>

        <div class="section-title">Key metrics</div>
        <div class="metric">Movement: {fmt_num(row.get(movement_col), 4) if movement_col else "N/A"}</div>
        <div class="metric">Hand/arm movement: {fmt_num(row.get(hand_col), 4) if hand_col else "N/A"}</div>
        <div class="metric">Non-neutral emotion: {fmt_num(row.get(emotion_col), 4) if emotion_col else "N/A"}</div>
        <div class="metric">Speech rate z-score: {fmt_num(row.get(speech_col), 2) if speech_col else "N/A"}</div>
        <div class="metric">Pitch variation z-score: {fmt_num(row.get(pitch_col), 2) if pitch_col else "N/A"}</div>
        <div class="metric">Interruptions / 100 speaking sec: {fmt_num(row.get(interrupt_col), 2) if interrupt_col else "N/A"}</div>

        <div class="section-title">Main themes</div>
        <div class="metric">{topics_html}</div>

        <div class="section-title">Emotion accumulation</div>
        <div class="metric">{emotions_html}</div>
    </div>
    """

cards_html += "</div>"

display(HTML(cards_html))

In [ ]:
# ============================================================
# CLEANED TOPIC × CANDIDATE EMOTION HEATMAP
# ============================================================

def wrap_label(label, width=22):
    return "\n".join(textwrap.wrap(str(label), width=width))


value_col = get_col(
    topic_profiles_clean,
    [
        "avg_non_neutral_emotion_score",
        "mean_non_neutral_emotion_score",
        "non_neutral_emotion_score",
    ]
)

if value_col is None:
    print("Could not find emotion score column.")
    print(topic_profiles_clean.columns.tolist())
else:
    heat = topic_profiles_clean.pivot_table(
        index="topic_clean",
        columns="candidate",
        values=value_col,
        aggfunc="mean",
    )

    # Remove moderator/no speech from columns if present
    heat = heat[[c for c in heat.columns if c not in ["Moderador/Other", "No speech"]]]

    # Keep only topics with enough information
    heat = heat.dropna(how="all")

    plt.figure(figsize=(13, max(5, 0.45 * len(heat.index))))
    plt.imshow(heat.fillna(0), aspect="auto")

    plt.title("Average non-neutral emotion by topic and candidate")
    plt.xlabel("Candidate")
    plt.ylabel("Topic")

    plt.xticks(
        range(len(heat.columns)),
        [wrap_label(c, 14) for c in heat.columns],
        rotation=45,
        ha="right",
    )

    plt.yticks(
        range(len(heat.index)),
        [wrap_label(t, 28) for t in heat.index],
    )

    plt.colorbar(label="Avg non-neutral emotion score")
    plt.tight_layout()
    plt.savefig(CLEAN_PLOT_DIR / "clean_topic_candidate_emotion_heatmap.png", dpi=220)
    plt.show()

In [ ]:
# ============================================================
# CLEANED TOPIC × CANDIDATE MOVEMENT HEATMAP
# ============================================================

move_col = get_col(
    topic_profiles_clean,
    [
        "avg_movement",
        "mean_movement",
        "movement_score",
    ]
)

if move_col is None:
    print("Could not find movement column.")
    print(topic_profiles_clean.columns.tolist())
else:
    heat_move = topic_profiles_clean.pivot_table(
        index="topic_clean",
        columns="candidate",
        values=move_col,
        aggfunc="mean",
    )

    heat_move = heat_move[[c for c in heat_move.columns if c not in ["Moderador/Other", "No speech"]]]
    heat_move = heat_move.dropna(how="all")

    plt.figure(figsize=(13, max(5, 0.45 * len(heat_move.index))))
    plt.imshow(heat_move.fillna(0), aspect="auto")

    plt.title("Average movement by topic and candidate")
    plt.xlabel("Candidate")
    plt.ylabel("Topic")

    plt.xticks(
        range(len(heat_move.columns)),
        [wrap_label(c, 14) for c in heat_move.columns],
        rotation=45,
        ha="right",
    )

    plt.yticks(
        range(len(heat_move.index)),
        [wrap_label(t, 28) for t in heat_move.index],
    )

    plt.colorbar(label="Avg movement score")
    plt.tight_layout()
    plt.savefig(CLEAN_PLOT_DIR / "clean_topic_candidate_movement_heatmap.png", dpi=220)
    plt.show()

In [ ]:
# ============================================================
# CLEANED TOP THEMES PER CANDIDATE TABLE
# ============================================================

theme_rows = []

for candidate in sorted(topic_profiles_clean["candidate"].dropna().unique()):
    if candidate in ["Moderador/Other", "No speech"]:
        continue

    sub = topic_profiles_clean[topic_profiles_clean["candidate"] == candidate].copy()

    if "total_topic_overlap_seconds" in sub.columns:
        ranked = (
            sub.groupby("topic_clean")["total_topic_overlap_seconds"]
            .sum()
            .sort_values(ascending=False)
            .head(5)
        )
    elif "n_windows" in sub.columns:
        ranked = (
            sub.groupby("topic_clean")["n_windows"]
            .sum()
            .sort_values(ascending=False)
            .head(5)
        )
    else:
        ranked = sub["topic_clean"].value_counts().head(5)

    for rank, (topic, value) in enumerate(ranked.items(), start=1):
        theme_rows.append(
            {
                "candidate": candidate,
                "rank": rank,
                "theme": topic,
                "score_or_seconds": value,
            }
        )

top_themes_clean = pd.DataFrame(theme_rows)

display(top_themes_clean)

top_themes_clean.to_csv(CLEAN_PLOT_DIR / "clean_top_themes_by_candidate.csv", index=False)
print("Saved:", CLEAN_PLOT_DIR / "clean_top_themes_by_candidate.csv")

In [ ]:
# ============================================================
# CANDIDATE PROFILE CARDS — CLEAN TOPICS FROM TOPIC FILE
# Fixes broken accents by NOT using the pre-compressed top_topics string.
# Instead it recomputes top topics from 04_candidate_topic_profiles.csv.
# ============================================================

import re
import html
import unicodedata
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

# ------------------------------------------------------------
# Paths / load data
# ------------------------------------------------------------

OUT_DIR = Path("outputs/multimodal_profile_analysis")
PLOT_DIR = Path("outputs/presentation_graphs")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

if "profiles_c" not in globals():
    profiles_c = pd.read_csv(OUT_DIR / "02_candidate_profiles_global.csv")

# Load topic profiles directly. This is the key fix.
topic_profiles_for_cards = pd.read_csv(OUT_DIR / "04_candidate_topic_profiles.csv")


# ------------------------------------------------------------
# General helpers
# ------------------------------------------------------------

def norm01(s):
    s = pd.to_numeric(s, errors="coerce")
    mn = s.min()
    mx = s.max()

    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(0.0, index=s.index)

    return (s - mn) / (mx - mn)


def metric(row, col, digits=2):
    if col not in row.index:
        return "n/a"

    x = row.get(col, np.nan)

    if pd.isna(x):
        return "n/a"

    try:
        if digits == 0:
            return f"{float(x):.0f}"
        return f"{float(x):.{digits}f}"
    except Exception:
        return str(x)


def strip_accents_for_matching(text):
    text = str(text)
    return "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )


def try_fix_mojibake(text):
    """
    Fix common UTF-8 read as Latin-1 problems, e.g.
    'JustiÃ§a' -> 'Justiça'
    'ProteÃ§Ã£o' -> 'Proteção'
    """
    if text is None or pd.isna(text):
        return text

    s = str(text)

    if "Ã" in s or "Â" in s:
        try:
            return s.encode("latin1").decode("utf-8")
        except Exception:
            return s

    return s


def clean_topic_name(topic):
    """
    Clean topic names while preserving Portuguese accents.
    Removes no-topic / unidentified-topic labels.
    """
    if topic is None or pd.isna(topic):
        return None

    original = try_fix_mojibake(str(topic)).strip()

    if original == "":
        return None

    # Remove bullets / numbering
    original = re.sub(r"^[•\-\d\.\)\s]+", "", original).strip()

    # Remove trailing numeric values, if any
    original = re.sub(r"\s*[:=]\s*[-+]?\d+(\.\d+)?%?$", "", original).strip()
    original = re.sub(r"\s*\([-+]?\d+(\.\d+)?%?\)$", "", original).strip()

    # Clean separators but keep accents
    display = (
        original
        .replace("_", " ")
        .replace("  ", " ")
        .strip()
    )

    key = strip_accents_for_matching(display).lower()
    key = key.replace("/", " ")
    key = key.replace("-", " ")
    key = re.sub(r"\s+", " ", key).strip()

    no_topic_patterns = [
        "nenhum",
        "nao identificado",
        "nao identificada",
        "sem tema",
        "sem topico",
        "no theme",
        "no topic",
        "unknown",
        "none",
        "nan",
    ]

    if any(p in key for p in no_topic_patterns):
        return None

    pretty_map = {
        "economia protecao social": "Economia / Proteção Social",
        "governo partidos": "Governo / Partidos",
        "eleicoes campanha": "Eleições / Campanha",
        "justica seguranca": "Justiça / Segurança",
        "saude": "Saúde",
        "educacao": "Educação",
        "internacional": "Internacional",
        "habitacao": "Habitação",
        "ambiente": "Ambiente",
        "cultura": "Cultura",
        "imigracao": "Imigração",
        "seguranca": "Segurança",
        "justica": "Justiça",
    }

    if key in pretty_map:
        return pretty_map[key]

    if "/" in display:
        return " / ".join(
            p.strip().capitalize()
            for p in display.split("/")
            if p.strip()
        )

    return display.capitalize()


# ------------------------------------------------------------
# Topic helpers — recompute top topics from topic_profiles file
# ------------------------------------------------------------

def find_topic_column(df):
    for col in ["topic", "dominant_topic", "Tema_Dominante", "theme"]:
        if col in df.columns:
            return col

    possible = [c for c in df.columns if "topic" in c.lower() or "tema" in c.lower()]
    if possible:
        return possible[0]

    raise ValueError("Could not find a topic/theme column in topic_profiles_for_cards.")


def find_topic_weight_column(df):
    candidates = [
        "total_topic_overlap_seconds",
        "dominant_topic_overlap_seconds",
        "topic_overlap_seconds",
        "speaking_seconds",
        "visible_seconds",
        "n_windows",
        "window_count",
    ]

    for col in candidates:
        if col in df.columns:
            return col

    return None


TOPIC_COL = find_topic_column(topic_profiles_for_cards)
TOPIC_WEIGHT_COL = find_topic_weight_column(topic_profiles_for_cards)

topic_cards_clean = topic_profiles_for_cards.copy()
topic_cards_clean["topic_clean"] = topic_cards_clean[TOPIC_COL].apply(clean_topic_name)
topic_cards_clean = topic_cards_clean[topic_cards_clean["topic_clean"].notna()].copy()

print("Using topic column:", TOPIC_COL)
print("Using topic weight column:", TOPIC_WEIGHT_COL)
print("Topics after cleaning:")
display(
    pd.DataFrame(
        sorted(topic_cards_clean["topic_clean"].dropna().unique()),
        columns=["topic_clean"]
    )
)


def top_topics_for_candidate(candidate, n=4):
    sub = topic_cards_clean[topic_cards_clean["candidate"] == candidate].copy()

    if len(sub) == 0:
        return []

    if TOPIC_WEIGHT_COL is not None:
        ranked = (
            sub.groupby("topic_clean")[TOPIC_WEIGHT_COL]
            .sum()
            .sort_values(ascending=False)
            .head(n)
        )
    else:
        ranked = (
            sub["topic_clean"]
            .value_counts()
            .head(n)
        )

    return [f"• {html.escape(str(topic))}" for topic in ranked.index.tolist()]


# ------------------------------------------------------------
# Emotion helper
# ------------------------------------------------------------

def top_items_from_string(raw, n=4):
    """
    Generic parser for top_emotions.
    """
    if raw is None or pd.isna(raw):
        return []

    s = try_fix_mojibake(str(raw)).strip()

    if s == "":
        return []

    parts = re.split(r";|\n|\|", s)
    out = []

    for part in parts:
        part = part.strip()

        if not part:
            continue

        part = re.sub(r"^[•\-\d\.\)\s]+", "", part).strip()
        part = part.replace("_", " ")

        out.append(f"• {html.escape(part)}")

    return out[:n]


def badge(text):
    return f"<span class='badge'>{html.escape(text)}</span>"


def mini_bar(label, value):
    value = 0 if pd.isna(value) else float(value)
    pct = max(0, min(100, value * 100))

    return f"""
    <div class='bar-row'>
      <div class='bar-label'>{html.escape(label)}</div>
      <div class='bar-track'><div class='bar-fill' style='width:{pct:.0f}%'></div></div>
      <div class='bar-num'>{pct:.0f}</div>
    </div>
    """


# ------------------------------------------------------------
# Build normalized style values
# ------------------------------------------------------------

STYLE_FEATURES = {
    "Movement": "avg_movement",
    "Hand gestures": "avg_hand_movement",
    "Emotion intensity": "avg_non_neutral_emotion_score",
    "Speech rate": "avg_speechrate_z",
    "Pitch variation": "avg_pitchvar_z",
    "Interruptions": "interruptions_per_100_speaking_sec",
}

STYLE_FEATURES = {
    label: col
    for label, col in STYLE_FEATURES.items()
    if col in profiles_c.columns
}

style_df = profiles_c[["candidate"] + list(STYLE_FEATURES.values())].copy()

for label, col in STYLE_FEATURES.items():
    style_df[label] = norm01(style_df[col])

style_norm = style_df[["candidate"] + list(STYLE_FEATURES.keys())].set_index("candidate")


def candidate_badges(style_row):
    out = []

    if style_row.get("Movement", 0) >= 0.70:
        out.append("high movement")

    if style_row.get("Hand gestures", 0) >= 0.70:
        out.append("gestural")

    if style_row.get("Emotion intensity", 0) >= 0.70:
        out.append("visually expressive")

    if style_row.get("Speech rate", 0) >= 0.70:
        out.append("fast speech")

    if style_row.get("Pitch variation", 0) >= 0.70:
        out.append("variable pitch")

    if style_row.get("Interruptions", 0) >= 0.70:
        out.append("interrupts often")

    if not out:
        out.append("more composed / balanced")

    return out


# ------------------------------------------------------------
# Create profile cards
# ------------------------------------------------------------

cards = []

for _, row in profiles_c.iterrows():
    cand = row["candidate"]

    if cand in ["Moderador/Other", "No speech"]:
        continue

    if cand not in style_norm.index:
        continue

    sr = style_norm.loc[cand]

    badges = " ".join(
        badge(x)
        for x in candidate_badges(sr)
    )

    bars = "".join(
        mini_bar(label, sr[label])
        for label in style_norm.columns
    )

    top_emotions = (
        "<br>".join(top_items_from_string(row.get("top_emotions", ""), 4))
        or "n/a"
    )

    # IMPORTANT: this now comes from 04_candidate_topic_profiles.csv,
    # not from the possibly corrupted top_topics string.
    top_topics = (
        "<br>".join(top_topics_for_candidate(cand, 4))
        or "n/a"
    )

    n_debates = (
        int(row.get("n_debates", 0))
        if pd.notna(row.get("n_debates", np.nan))
        else "n/a"
    )

    cards.append(f"""
    <div class='profile-card'>
      <h3>{html.escape(str(cand))}</h3>
      <div>{badges}</div>

      <div class='quick-stats'>
        <div><b>Debates</b><br>{n_debates}</div>
        <div><b>Speaking sec.</b><br>{metric(row, 'total_speaking_seconds', 0)}</div>
        <div><b>Visible sec.</b><br>{metric(row, 'total_visible_seconds', 0)}</div>
      </div>

      {bars}

      <div class='two-col'>
        <div><b>Top emotions</b><br>{top_emotions}</div>
        <div><b>Top topics</b><br>{top_topics}</div>
      </div>
    </div>
    """)


html_out = f"""
<style>
.profile-grid {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(320px, 1fr));
    gap: 16px;
}}

.profile-card {{
    border: 1px solid #ddd;
    border-radius: 14px;
    padding: 16px;
    box-shadow: 0 2px 8px rgba(0,0,0,.08);
    background: white;
}}

.profile-card h3 {{
    margin: 0 0 8px 0;
}}

.badge {{
    display: inline-block;
    border: 1px solid #bbb;
    border-radius: 999px;
    padding: 3px 8px;
    margin: 2px;
    font-size: 12px;
    background: #f7f7f7;
}}

.quick-stats {{
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 8px;
    margin: 12px 0;
    font-size: 13px;
}}

.quick-stats div {{
    border-radius: 10px;
    background: #f6f6f6;
    padding: 8px;
}}

.bar-row {{
    display: grid;
    grid-template-columns: 110px 1fr 34px;
    align-items: center;
    gap: 8px;
    margin: 5px 0;
    font-size: 12px;
}}

.bar-track {{
    background: #e9e9e9;
    border-radius: 999px;
    height: 10px;
    overflow: hidden;
}}

.bar-fill {{
    background: #777;
    height: 100%;
    border-radius: 999px;
}}

.bar-num {{
    text-align: right;
    color: #555;
}}

.two-col {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 12px;
    margin-top: 12px;
    font-size: 13px;
    line-height: 1.35;
}}
</style>

<h2>Candidate debating style profiles</h2>
<p style="font-size:13px; color:#555;">
Top topics are recomputed from the topic-profile table, with “Nenhum / Não identificado” removed.
</p>

<div class='profile-grid'>
{''.join(cards)}
</div>
"""

display(HTML(html_out))

html_path = PLOT_DIR / "candidate_profile_cards_clean_topics_from_topic_file.html"
html_path.write_text(
    f"<html><head><meta charset='utf-8'></head><body><h1>Candidate debating style profiles</h1>{html_out}</body></html>",
    encoding="utf-8",
)

print("Saved profile-card HTML:", html_path)